# Reproducing Knierim, McNaughton & Poe (2000) from dandiset 001754

Paper. Knierim JJ, McNaughton BL, Poe GR (2000). Three-dimensional spatial selectivity of
hippocampal neurons during space flight. *Nature Neuroscience* 3:209-210.
[doi:10.1038/72910](https://doi.org/10.1038/72910). The paper is not open access. This attempt
used only its abstract and the titles of its three figures, from the publisher's page. Three
claims come from the dandiset's own description, which summarises the paper's findings; they
could not be checked against the paper's text.

Dandiset. [DANDI:001754](https://dandiarchive.org/dandiset/001754/0.260728.1352), version
0.260728.1352. The paper (2000) cites no DANDI version. Three versions are published
(0.260227.0029, 0.260227.0031, 0.260728.1352); they hold the same eight files (same blobs, checked
below) and differ only in metadata (the last adds the journal name to the paper's link). The most
recent is pinned.

What the dandiset holds. Three rats on the Neurolab mission (STS-90, April 1998): a preflight
recording of each, flight day 4 (FD4) of each, and flight day 9 (FD9) of Rats 1 and 2. Each file
has sorted units, task epochs (BL baseline, ES Escher staircase, MC Magic Carpet), and the 64 x
64 rate and occupancy maps of the original analysis for ES and MC. The flight files also hold the
video position, but only at spike times. There is no EEG and no head direction.

Conversion code read. [catalystneuro/mcnaughton2000-to-nwb](https://github.com/catalystneuro/mcnaughton2000-to-nwb)
at commit b538d9e07079394362a64e5118a226e91bdd7ab5, linked from the dandiset. It merges the
per-epoch spike files (.CEL) of each tetrode and cluster id into one unit, takes position from
the spike files' position columns, builds the epochs from the spike files' headers and names, and
reads the .RMA rate maps. The paper's own analysis code is not linked and was not found.

| Claim | Source | Tested |
|---|---|---|
| `flight-spatial-code`: a robust spatial code on the Escher staircase in flight | Abstract | Yes |
| `three-orthogonal-surfaces`: the code covers each of the three surfaces | Abstract | Blocked: metadata insufficient |
| `si-comparable-to-preflight`: flight spatial information comparable to preflight | Dandiset description | Yes |
| `fd4-abnormal-selectivity`: abnormal selectivity on flight day 4 | Dandiset description | Yes |
| `fig3-stability-across-sessions`: place fields stable across sessions | Figure 3 title | Yes |
| `theta-normal`: normal theta in flight | Dandiset description | Blocked: data missing |
| `ripples-normal`: normal sharp waves and ripples in flight | Dandiset description | Blocked: data missing |

In [ ]:
import time
import warnings

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import requests
from scipy import stats

import neurolab as nl

pd.set_option("display.width", 200)
pd.set_option("display.max_columns", 30)
pd.set_option("display.max_rows", 300)
warnings.filterwarnings("ignore", category=RuntimeWarning)

started = time.time()
sessions = nl.load_all()
by_label = {nl.label(s): s for s in sessions}
print(f"loaded {len(sessions)} files from dandiset {nl.DANDISET_ID} version {nl.DANDISET_VERSION} "
      f"in {time.time() - started:.0f} s")

## Data check

Each check of the metadata against the values is stated under its output.

In [ ]:
api = "https://api.dandiarchive.org/api/dandisets/001754/versions"
versions = [v["version"] for v in requests.get(f"{api}/", timeout=60).json()["results"] if v["version"] != "draft"]
blobs = {}
for v in versions:
    assets = requests.get(f"{api}/{v}/assets/", params={"page_size": 100}, timeout=60).json()["results"]
    blobs[v] = sorted((a["path"], a["size"], a.get("blob")) for a in assets)
print("Published versions:", versions)
print("All versions hold the same files with the same blobs:", all(b == blobs[versions[0]] for b in blobs.values()))
for path, size, _ in blobs[nl.DANDISET_VERSION]:
    print(f"  {path}  {size / 1e6:.1f} MB")

print("\nEverything the largest flight file holds (datasets, shape, type, description):")
flight = [p for p in nl.asset_urls() if "Rat1_ses-19980425" in p[0]][0]
for line in nl.h5_tree(flight):
    if "/rate_maps/" in line or line.startswith(("units/", "intervals/", "processing/behavior", "general/subject", "general/extracellular_ephys/electrodes/")) :
        print(" ", line)

The three published versions hold the same eight files. The listing shows every table and its
columns: the units table has only `tetrode`, `cluster_id`, `spike_times` and `electrodes`; the
epochs table `session_type` and its description; the rate-map table `tetrode`, `source_file`,
`session_type`, `cell_number`, `rate_map` and `occupancy_map`; the position one series of x and y
in pixels. The conversion code's README also lists `session_type` and `source_file` as unit
columns; they are not in the files, so a unit cannot be traced to the spike file it came from.

In [ ]:
rows = []
for s in sessions:
    ep = s["epochs"]
    rows.append({"file": s["path"].split("/")[1][:31], "rat": s["subject"], "day": s["day"], "start": s["start"][:16],
                 "units": len(s["units"]), "spikes": int(sum(len(x) for x in s["units"]["spikes"])),
                 "tetrodes": s["units"]["tetrode"].nunique(),
                 "epochs": " ".join(f"{t}[{a:.0f}-{b:.0f}]" for a, b, t in ep[["start", "stop", "type"]].itertuples(index=False)),
                 "rate_maps": len(s["maps"]), "map_types": "/".join(sorted(s["maps"]["type"].unique())),
                 "position_samples": len(s["t"]) if "t" in s else 0})
info = pd.DataFrame(rows)
print(info.to_string(index=False))
print("\nFiles per rat and day:"); print(pd.crosstab(info.rat, info.day).to_string())
for s in sessions:
    print(f"{nl.label(s):14s} {s['description']}")

Three rats, eight files: preflight and FD4 for each rat, FD9 for Rats 1 and 2, as the description
says. Rate maps exist only for ES and MC epochs, not for baseline. Position exists only in the five
flight files. In the Rat1 FD4 file two baseline epochs overlap (2525 to 3390 s and 2720 to 3390 s);
these are two spike-file headers and are not used here.

In [ ]:
print("Position, flight files. Stated unit:", {s['xy_unit'] for s in sessions if 'xy' in s}, "; description:", {s['xy_description'] for s in sessions if 'xy' in s})
rows = []
for s in sessions:
    if "xy" not in s:
        continue
    xy, v, t = s["xy"], s["valid"], s["t"]
    all_spikes = np.unique(np.concatenate(list(s["units"]["spikes"])))
    rows.append({"session": nl.label(s), "samples": len(t), "missing_(0,255)": round(1 - v.mean(), 3),
                 "x_min": xy[v, 0].min(), "x_max": xy[v, 0].max(), "y_min": xy[v, 1].min(), "y_max": xy[v, 1].max(),
                 "t_first": t[0], "t_last": t[-1], "median_dt_s": round(float(np.median(np.diff(t))), 3),
                 "frac_samples_at_spike_times": np.isin(t, all_spikes).mean(),
                 "frac_spike_times_with_sample": round(float(np.isin(all_spikes, t).mean()), 4)})
print(pd.DataFrame(rows).round(2).to_string(index=False))

Position is in video pixels from 0 to 255 in x and y. A sample at exactly (0, 255) is the tracker's
lost position (5 to 12% of samples); these are dropped. Every position sample is at a spike time,
so the trace is sampled only when some unit fires, at irregular intervals; consistent with the
description ("sampled at spike occurrence times"). No continuous position, and so no occupancy
time, can be built from it; the occupancy maps of the original analysis are used instead.

In [ ]:
print("Do the Rat1 and Rat2 files of the same flight day hold the same position trace?")
for day in ["FD4", "FD9"]:
    a, b = by_label[f"Rat1 {day}"], by_label[f"Rat2 {day}"]
    common, ia, ib = np.intersect1d(a["t"], b["t"], return_indices=True)
    both = a["valid"][ia] & b["valid"][ib]
    dist = np.hypot(*(a["xy"][ia][both] - b["xy"][ib][both]).T)
    print(f"  {day}: {len(common)} identical timestamps; at {both.sum()} with both valid, positions within 2 px: "
          f"{(dist < 2).mean():.3f}, median distance {np.median(dist):.2f} px; the same epoch boundaries: "
          f"{set(a['epochs'].start) == set(b['epochs'].start)} (starts in one file only: {sorted(set(a['epochs'].start) ^ set(b['epochs'].start))})")
    print("    Rat1 epochs:", list(a["epochs"].itertuples(index=False, name=None)))
    print("    Rat2 epochs:", list(b["epochs"].itertuples(index=False, name=None)))

The Rat1 and Rat2 files of each flight day were recorded together and share one position trace:
at timestamps they have in common, 99% of positions agree within 2 pixels. One video position
cannot be the position of two rats on different apparatus, so in any window the trace belongs to
at most one of them. The epochs show that the rats took turns: when one rat is labelled ES or MC,
the other is labelled BL. The next check asks whose position the trace is in each task epoch.

In [ ]:
rows = []
for s in sessions:
    if "xy" not in s:
        continue
    for a, b, typ in s["epochs"][["start", "stop", "type"]].itertuples(index=False):
        if typ == "BL":
            continue
        occ, scores = nl.epoch_occupancy(s, a, b, typ)
        other = [nl.epoch_occupancy(s, a, b, t)[1][0][0] for t in {"ES", "MC"} - {typ}]
        rows.append({"session": nl.label(s), "epoch": f"{typ} {a:.0f}-{b:.0f}", "duration_s": b - a,
                     "match_own_type": round(scores[0][0], 2), "match_other_type": round(other[0], 2),
                     "other_candidates_same_type": [round(r, 2) for r, _ in scores[1:]],
                     "occupancy_counts": scores[0][1], "counts_per_s": round(scores[0][1] / (b - a), 1)})
print("Correlation of the binary occupancy map with the binary histogram of the epoch's position samples:")
print(pd.DataFrame(rows).to_string(index=False))

For each ES and MC epoch, the position samples in the epoch cover the bins of the occupancy map of
the same type better than those of the other type (0.46 to 0.88 against 0.29 to 0.55), and for
Rat3's two ES epochs each matches a different one of its two ES occupancy maps. The occupancy
totals are 38 to 57 counts per second of the epoch, the same order for every epoch (the video
rate is not stated). The epoch labels, the epoch times and the shared position trace are therefore
consistent with the original analysis: in each labelled task epoch the trace is the position of
the rat on the task. This is a check of position against occupancy; whether spikes and position
share a clock is shown with spike-based maps below.

In [ ]:
print("Rate-map values:")
r_all = np.concatenate([s["rate_maps"].ravel() for s in sessions])
print("  negative values:", np.unique(r_all[r_all < 0]), "; fraction of bins:", round(float((r_all < 0).mean()), 3))
print("  -999 only where occupancy is 0:", all((s["rate_maps"][s["occupancy_maps"] > 0] >= 0).all() for s in sessions))
print("  max rate per file (Hz):", {nl.label(s): round(float(s["rate_maps"].max()), 1) for s in sessions})
print("\nOccupancy in bin [63, 0], the bin of the lost position (0, 255), as a fraction of each occupancy map:")
for s in sessions:
    print("  ", nl.label(s), [f"{typ} {occ[nl.SENTINEL_BIN] / occ.sum():.3f}" for typ, occ, _ in nl.map_occupancy_groups(s)])

maps = nl.rma_table(sessions)
print("\nRate maps per file and type: all, identical copies of another map in the same file, names with CEL:")
print(maps.groupby(["rat", "day", "type"]).agg(maps=("info", "size"), duplicates=("duplicate", "sum"), cel_names=("cel_name", "sum")).to_string())
print("\nExamples of source_file names:", sorted(set(maps.source_file))[:30])
dups = []
for s in sessions:
    keys = {}
    for i, m in s["maps"].iterrows():
        keys.setdefault(s["rate_maps"][i].tobytes(), []).append(f"{m.tetrode}/{m.source_file}")
    dups += [(nl.label(s), v) for v in keys.values() if len(v) > 1]
print("\nIdentical rate maps within a file:")
for d in dups:
    print("  ", d)

The rate maps use -999 for unvisited bins; the table describes the column only as a rate in Hz.
The lost-position value (0, 255) was counted in the original occupancy (bin [63, 0], up to 19% of
an occupancy map in flight, under 2% preflight); that bin is left out here. Six groups of maps are identical copies within one
file, some under different tetrodes (for example Rat1 FD4 TT1 and TT11 `ESCELL~1.RMA`); each is
counted once. The `source_file` names are Windows 8.3 short names: `ESCELL~1` to `ESCELL~4` are the
first four files whose long names begin `EScell`, and names like `ES2BC0~1` are the hashed short
names Windows gives from the fifth on. So `cell_number`, read from the `~N` suffix, is the order
of the short name and not a cell number. The next check tests whether it matches the cluster id
nevertheless.

In [ ]:
# For each flight unit and task epoch, the dandiset rate map on the same tetrode and type whose
# spike-count prediction (rate x occupancy) correlates best with the unit's spike-count map.
rows = []
for s in sessions:
    if "xy" not in s:
        continue
    lookup = nl.PositionLookup(s)
    for a, b, typ in s["epochs"][["start", "stop", "type"]].itertuples(index=False):
        if typ == "BL":
            continue
        for _, u in s["units"].iterrows():
            sp = u.spikes[(u.spikes >= a) & (u.spikes <= b)]
            found, r, c = lookup(sp)
            if found.sum() < nl.MIN_SPIKES:
                continue
            counts = nl.count_map(r, c)
            cand = s["maps"][(s["maps"].tetrode == u.tetrode) & (s["maps"].type == typ)]
            best = max(((np.corrcoef((s["rate_maps"][i].clip(0) * nl.clean_occupancy(s["occupancy_maps"][i])).ravel(), counts.ravel())[0, 1],
                         s["maps"].loc[i, "cell_number"], s["maps"].loc[i, "source_file"]) for i in cand.index), default=(np.nan, None, None))
            transposed = max((np.corrcoef((s["rate_maps"][i].clip(0) * nl.clean_occupancy(s["occupancy_maps"][i])).ravel(), counts.T.ravel())[0, 1]
                              for i in cand.index), default=np.nan)
            rows.append({"session": nl.label(s), "type": typ, "tetrode": u.tetrode, "cluster": u.cluster, "spikes": int(found.sum()),
                         "best_r": best[0], "best_r_x_y_swapped": transposed, "best_cell_number": best[1], "best_file": best[2]})
link = pd.DataFrame(rows)
link["cluster_matches_cell_number"] = link.cluster == link.best_cell_number
print(f"Unit-epochs with at least {nl.MIN_SPIKES} spikes at a known position: {len(link)}")
print(link.groupby(["session", "type"]).agg(n=("best_r", "size"), median_best_r=("best_r", "median"),
      median_r_swapped=("best_r_x_y_swapped", "median"), cluster_eq_cell_number=("cluster_matches_cell_number", "mean")).round(2).to_string())
print("\nAll: median best r", round(link.best_r.median(), 2), "; best map has cell_number equal to the cluster id in",
      f"{link.cluster_matches_cell_number.sum()} of {len(link)}; best r > 0.5 in {(link.best_r > 0.5).sum()}")

With position binned as x to columns and y to rows at 4 pixels per bin, each unit's spike-count map
correlates with a rate map of its tetrode: the median best r per epoch is 0.21 to 0.77, and 0.00 to
0.15 with x and y swapped. This supports the binning used here and shows that the original maps were
made from the same spikes and positions. The best-matching map has `cell_number` equal to the cluster
id for 84 of 211 unit-epochs (0.88 in Rat1 FD9 ES; 0 in Rat3 FD4 ES, whose `ES1CEL~N` names give no
`cell_number`), so `cell_number` does not identify the unit of a rate map, contrary to the rate-map
table's description. The rate maps are therefore used on their own, and the spike-based analysis
uses the units.

In [ ]:
rows = []
for s in sessions:
    ep = s["epochs"]
    for _, u in s["units"].iterrows():
        sp = u.spikes
        inside = np.zeros(len(sp), bool)
        for a, b in zip(ep.start, ep.stop):
            inside |= (sp >= a) & (sp <= b)
        isi = np.diff(sp)
        n_types = sum(((sp >= a) & (sp <= b)).any() for a, b in zip(ep.start, ep.stop))
        rows.append({"session": nl.label(s), "spikes": len(sp), "outside_epochs": int((~inside).sum()),
                     "zero_isi": int((isi == 0).sum()), "intervals": len(isi), "sorted": bool((isi >= 0).all()), "epochs_with_spikes": n_types})
sp_table = pd.DataFrame(rows)
res = {nl.label(s): sorted({round(float(np.min(np.diff(np.unique(np.round(np.concatenate(list(s['units'].spikes)), 6))))), 4)}) for s in sessions}
print("Smallest step between distinct spike times, per file (s):", res)
print(sp_table.groupby("session").agg(units=("spikes", "size"), spikes=("spikes", "sum"), outside_epochs=("outside_epochs", "sum"),
      zero_isi=("zero_isi", "sum"), intervals=("intervals", "sum"), all_sorted=("sorted", "all"),
      median_epochs_with_spikes=("epochs_with_spikes", "median")).assign(zero_isi_fraction=lambda d: (d.zero_isi / d.intervals).round(3)).to_string())

Spike times are sorted. They are given to 10 ms in five files and to 1 ms or finer in three; 2 to
15% of the intervals within a unit are zero (printed), which at this resolution can be two spikes in one step, or the same
spike from two overlapping epoch files merged into one unit. A unit spans several epochs because
the conversion merged the per-epoch spike files with the same tetrode and cluster id. Whether a
cluster id names the same neuron in different epochs is not stated; here it is an assumption,
needed only for the stability claim, and every other result is computed per epoch.

In [ ]:
# Clock and attribution contrast: spatial information of each rat's units in its own task epoch
# against the other rat's task epoch on the same day, where the shared position is the other rat's.
rng_check = np.random.default_rng(1754)
rows = []
for day in ["FD4", "FD9"]:
    for rat, other in [("Rat1", "Rat2"), ("Rat2", "Rat1")]:
        s, o = by_label[f"{rat} {day}"], by_label[f"{other} {day}"]
        for owner, src in [(rat, s), (other, o)]:
            for a, b, typ in src["epochs"][["start", "stop", "type"]].itertuples(index=False):
                if typ != "ES":
                    continue
                # occupancy from the file whose rat was on the task in this window
                occ, _ = nl.epoch_occupancy(src, a, b, typ)
                s_tmp = dict(s, maps=src["maps"], occupancy_maps=src["occupancy_maps"], rate_maps=src["rate_maps"])
                df = nl.epoch_units(s_tmp, a, b, typ, rng_check, n_shuffles=200)
                d = df.dropna(subset=["info"])
                rows.append({"day": day, "units_of": rat, "ES_epoch_of": owner, "window": f"{a:.0f}-{b:.0f}",
                             "units": len(d), "significant": int((d.p < 0.05).sum()),
                             "fraction": round(float((d.p < 0.05).mean()), 2) if len(d) else np.nan,
                             "median_info": round(float(d["info"].median()), 3) if len(d) else np.nan})
contrast = pd.DataFrame(rows)
print(contrast.to_string(index=False))

This contrast tests whether spikes and position share a clock and whose position the trace is.
Each rat's units are tested (200 shuffles) in its own ES epoch and in the other rat's ES epoch of the
same day, with the occupancy of the rat labelled as on the task. Rat1's units are selective in their
own ES epochs (10 of 20 on FD4, 12 of 25 on FD9, against 5% expected by chance), so for Rat1 spikes
and position share a clock. Rat2's units are no more selective than chance in their own ES epochs (0
of 16 on FD4, 3 of 28 on FD9) or in Rat1's (0 of 15, 0 of 41). Rat1's units are also selective in
Rat2's FD4 ES epoch (7 of 15), when Rat1 is labelled as on baseline, but not in Rat2's FD9 ES epoch
(1 of 36). So a shared clock could not be established for Rat2's spikes and the position, and in
Rat2's FD4 ES epoch the trace is consistent with Rat1's position. The data cannot show whose position
the trace is in Rat2's ES epochs. The original rate maps of Rat2 were made from the same trace (see
the previous check).

## flight-spatial-code

> Hippocampal neurons recorded on the space shuttle form a robust spatial code on the
> three-dimensional Escher staircase track in weightlessness. (Abstract)

Criterion (from the claims file): per in-flight ES epoch, a unit is significant when its spatial
information exceeds the 95th percentile of 500 circular-shift shuffles. Judged on the two FD9
sessions: reproduced when the fraction of significant units is at least 0.5 in both.

In [ ]:
rng_c1 = np.random.default_rng(20000301)
results = []
for s in sessions:
    if "xy" not in s:
        continue
    for a, b, typ in s["epochs"][["start", "stop", "type"]].itertuples(index=False):
        if typ in ("ES", "MC"):
            df = nl.epoch_units(s, a, b, typ, rng_c1)
            df["type"] = typ
            df["epoch"] = f"{typ} {a:.0f}-{b:.0f}"
            results.append(df)
units = pd.concat(results, ignore_index=True)
tested = units.dropna(subset=["info"]).copy()
tested["sig"] = tested.p < 0.05

def summary(d):
    n, k = len(d), int(d.sig.sum())
    return pd.Series({"units_in_file": np.nan, "tested": n, "significant": k, "fraction": round(k / n, 3) if n else np.nan,
                      "binom_p_vs_0.05": stats.binomtest(k, n, 0.05, alternative="greater").pvalue if n else np.nan,
                      "median_info": round(d["info"].median(), 3), "median_null95": round(d["null95"].median(), 3)})

per = tested.groupby(["type", "rat", "day", "epoch"]).apply(summary)
per["units_in_file"] = units.groupby(["type", "rat", "day", "epoch"]).size()
per["dropped_<50_spikes"] = per["units_in_file"] - per["tested"]
print(per.to_string())

In [ ]:
es = tested[tested.type == "ES"]
print("Judged sessions (FD9 ES):")
for rat in ["Rat1", "Rat2"]:
    d = es[(es.rat == rat) & (es.day == "FD9")]
    print(f"  {rat} FD9: {d.sig.sum()} of {len(d)} significant, fraction {d.sig.mean():.3f}, "
          f"binomial p {stats.binomtest(int(d.sig.sum()), len(d), 0.05, alternative='greater').pvalue:.2e}")
print("\nThreshold variants, fraction of significant ES units by session:")
var = []
for (rat, day, epoch), d in es.groupby(["rat", "day", "epoch"]):
    row = {"session": f"{rat} {day} {epoch}", "n": len(d)}
    for alpha in [0.05, 0.01]:
        row[f"p<{alpha}"] = round(float((d.p < alpha).mean()), 3)
    big = d[d.spikes_used >= 100]
    row["n_>=100_spikes"] = len(big)
    row["p<0.05,>=100_spikes"] = round(float((big.p < 0.05).mean()), 3) if len(big) else np.nan
    row["info>0.5_bits"] = round(float((d["info"] > 0.5).mean()), 3)
    var.append(row)
print(pd.DataFrame(var).to_string(index=False))
print("\nSpatial information of tested ES units, range:", round(es["info"].min(), 3), "to", round(es["info"].max(), 3), "bits/spike")

fig, ax = plt.subplots(figsize=(8, 3.5))
labels = []
for i, ((rat, day, epoch), d) in enumerate(es.groupby(["rat", "day", "epoch"])):
    ax.scatter(np.full(len(d), i) + np.random.default_rng(i).uniform(-0.15, 0.15, len(d)), d["info"], s=10,
               c=np.where(d.sig, "tab:blue", "tab:gray"))
    ax.scatter(np.full(len(d), i) + 0.3, d["null95"], s=6, marker="_", c="k")
    labels.append(f"{rat} {day}\n{epoch.split()[1]}")
ax.set_xticks(range(len(labels)), labels, fontsize=8)
ax.set_ylabel("spatial information (bits/spike)")
ax.set_title("ES units: blue significant, grey not; dashes: 95th percentile of shuffles")
plt.tight_layout()
plt.show()

In [ ]:
print("Attribution contrast from the data check, ES epochs:")
print(contrast.to_string(index=False))

**Result: partially reproduced.** On flight day 9, 13 of 25 Rat1 units (0.52) have significant
spatial information on the Escher staircase (binomial p = 3.6e-11 against 0.05), but only 3 of 28
Rat2 units (0.107, p = 0.16), which is not above chance; the claim holds for one of the two judged
sessions. On flight day 4, 10 of 20 Rat1 units, 0 of 16 Rat2 units, and 9 of 16 and 2 of 19 Rat3
units in its two ES epochs are significant. The Rat1 FD9 result holds at p < 0.01 (0.40) and with
100 spikes or more (0.54). Rat2's units are not selective with the shared position trace in any
window, and in Rat2's FD4 ES epoch the trace behaves as Rat1's position (7 of 15 Rat1 units
significant), so Rat2's result may come from a position trace that is not Rat2's. In the MC epochs,
which the claim does not cover, 0 to 2 units per session are significant; their 95th percentiles of
the shuffles are higher (median 3.06 to 5.74 bits/spike against 2.16 to 3.36 for ES).

## three-orthogonal-surfaces

> The spatial code covers each of the three orthogonal surfaces of the Escher staircase. (Abstract)

Blocked. The dandiset holds one two-dimensional pixel position from one camera; the NASA record
of the study mentions two synchronised cameras. Nothing in the files or the metadata describes
the camera's view or the track layout, so a pixel cannot be assigned to one of the three surfaces.

## si-comparable-to-preflight

> In flight, spatial information content of place cells is comparable to preflight levels.
> (Dandiset description)

Criterion: two-sided Mann-Whitney U test at alpha 0.05 of the spatial information of the
dandiset's rate maps, FD9 ES against preflight ES, Rats 1 and 2 pooled, every distinct map with a
positive mean rate. Reproduced when the test finds no difference.

In [ ]:
m = maps[(~maps.duplicate) & (maps.mean_rate > 0)].copy()
print("Maps dropped: duplicates", int(maps.duplicate.sum()), "; zero mean rate", int(((~maps.duplicate) & (maps.mean_rate <= 0)).sum()),
      "; kept", len(m), "of", len(maps))
print("Spatial information of rate maps, range:", round(m["info"].min(), 3), "to", round(m["info"].max(), 3), "bits/spike")
print(m.groupby(["type", "rat", "day"])["info"].describe()[["count", "50%", "25%", "75%", "min", "max"]].round(3).to_string())

def compare(a, b, alternative="two-sided"):
    if len(a) == 0 or len(b) == 0:
        return {"n_a": len(a), "n_b": len(b)}
    u = stats.mannwhitneyu(a, b, alternative=alternative)
    return {"n_a": len(a), "n_b": len(b), "median_a": round(float(np.median(a)), 3), "median_b": round(float(np.median(b)), 3),
            "diff": round(float(np.median(a) - np.median(b)), 3), "ratio": round(float(np.median(a) / np.median(b)), 2),
            "U": float(u.statistic), "p": float(u.pvalue), "rank_biserial": round(1 - 2 * u.statistic / (len(a) * len(b)), 3)}

sel = lambda d, typ, days, rats: d[(d.type == typ) & d.day.isin(days) & d.rat.isin(rats)]["info"].values
main = compare(sel(m, "ES", ["FD9"], ["Rat1", "Rat2"]), sel(m, "ES", ["preflight"], ["Rat1", "Rat2"]))
print("\nMain test, ES FD9 (a) against ES preflight (b), Rats 1 and 2:", main)

In [ ]:
rows = []
variants = {
    "per rat Rat1": (m, "ES", ["FD9"], ["Rat1"], ["preflight"], ["Rat1"]),
    "per rat Rat2": (m, "ES", ["FD9"], ["Rat2"], ["preflight"], ["Rat2"]),
    "CEL names only": (m[m.cel_name], "ES", ["FD9"], ["Rat1", "Rat2"], ["preflight"], ["Rat1", "Rat2"]),
    "peak >= 1 Hz": (m[m.peak_rate >= 1], "ES", ["FD9"], ["Rat1", "Rat2"], ["preflight"], ["Rat1", "Rat2"]),
    "MC maps": (m, "MC", ["FD9"], ["Rat1", "Rat2"], ["preflight"], ["Rat1", "Rat2"]),
    "all flight ES (FD4+FD9, 3 rats)": (m, "ES", ["FD4", "FD9"], ["Rat1", "Rat2", "Rat3"], ["preflight"], ["Rat1", "Rat2", "Rat3"]),
}
for name, (d, typ, da, ra, db, rb) in variants.items():
    a = d[(d.type == typ) & d.day.isin(da) & d.rat.isin(ra)]["info"].values
    b = d[(d.type == typ) & d.day.isin(db) & d.rat.isin(rb)]["info"].values
    rows.append({"variant": name} | compare(a, b))
print("Variants (a = flight, b = preflight):")
print(pd.DataFrame(rows).to_string(index=False))

fig, ax = plt.subplots(figsize=(7, 3.5))
groups = [(f"{rat}\n{day}", m[(m.type == "ES") & (m.rat == rat) & (m.day == day)]["info"].values)
          for rat in ["Rat1", "Rat2", "Rat3"] for day in ["preflight", "FD4", "FD9"]]
groups = [(g, v) for g, v in groups if len(v)]
ax.boxplot([v for _, v in groups], showfliers=True)
ax.set_xticks(range(1, len(groups) + 1), [g for g, _ in groups], fontsize=8)
ax.set_ylabel("spatial information of ES rate maps (bits/spike)")
plt.tight_layout()
plt.show()

**Result: reproduced.** Flight day 9 ES maps (64, Rats 1 and 2) have a median of 1.385 bits/spike
against 1.448 for preflight ES maps (53); Mann-Whitney p = 0.45, a difference of -0.064 bits/spike
(ratio 0.96, rank-biserial 0.081). Every variant gives p of 0.20 or more: per rat, maps named CEL
only, maps with a peak of 1 Hz or more, MC maps, and all flight maps. These are the spatial
information values of the original rate maps, with no correction for the number of spikes. Rat2's
maps rest on a position trace that could not be attributed to Rat2: its FD9 ES maps have a median of
1.343 bits/spike, close to Rat1's, while only 3 of its 28 units are significant against shuffles.

## fd4-abnormal-selectivity

> On the first in-flight exposures (flight day 4) spatial selectivity was abnormal. (Dandiset
> description)

Criterion: abnormal is read as lower spatial information. FD4 ES rate maps (Rats 1 to 3) against
preflight ES rate maps (Rats 1 to 3); reproduced when the FD4 median is lower and a one-sided
Mann-Whitney U test gives p < 0.05.

In [ ]:
fd4 = sel(m, "ES", ["FD4"], ["Rat1", "Rat2", "Rat3"])
pre = sel(m, "ES", ["preflight"], ["Rat1", "Rat2", "Rat3"])
main4 = compare(fd4, pre, alternative="less")
print("Main test, ES FD4 (a) below ES preflight (b), Rats 1 to 3, one-sided:", main4)
rows = []
for rat in ["Rat1", "Rat2", "Rat3"]:
    rows.append({"comparison": f"{rat} FD4 < preflight"} | compare(sel(m, "ES", ["FD4"], [rat]), sel(m, "ES", ["preflight"], [rat]), "less"))
for rat in ["Rat1", "Rat2"]:
    rows.append({"comparison": f"{rat} FD4 < FD9"} | compare(sel(m, "ES", ["FD4"], [rat]), sel(m, "ES", ["FD9"], [rat]), "less"))
rows.append({"comparison": "Rats 1-2 FD4 < FD9"} | compare(sel(m, "ES", ["FD4"], ["Rat1", "Rat2"]), sel(m, "ES", ["FD9"], ["Rat1", "Rat2"]), "less"))
rows.append({"comparison": "CEL names, FD4 < preflight"} | compare(sel(m[m.cel_name], "ES", ["FD4"], ["Rat1", "Rat2", "Rat3"]), sel(m[m.cel_name], "ES", ["preflight"], ["Rat1", "Rat2", "Rat3"]), "less"))
rows.append({"comparison": "peak >= 1 Hz, FD4 < preflight"} | compare(sel(m[m.peak_rate >= 1], "ES", ["FD4"], ["Rat1", "Rat2", "Rat3"]), sel(m[m.peak_rate >= 1], "ES", ["preflight"], ["Rat1", "Rat2", "Rat3"]), "less"))
rows.append({"comparison": "MC maps, FD4 < preflight"} | compare(sel(m, "MC", ["FD4"], ["Rat1", "Rat2", "Rat3"]), sel(m, "MC", ["preflight"], ["Rat1", "Rat2", "Rat3"]), "less"))
print(pd.DataFrame(rows).to_string(index=False))

print("\nSpike-based fraction of significant ES units by day (from flight-spatial-code):")
print(es.groupby(["day", "rat"]).agg(tested=("sig", "size"), significant=("sig", "sum"), fraction=("sig", "mean"),
      median_info=("info", "median")).round(3).to_string())
k4, n4 = int(es[es.day == "FD4"].sig.sum()), int((es.day == "FD4").sum())
k9, n9 = int(es[es.day == "FD9"].sig.sum()), int((es.day == "FD9").sum())
print(f"FD4 {k4}/{n4} against FD9 {k9}/{n9}; Fisher exact one-sided (FD4 lower) p = "
      f"{stats.fisher_exact([[k4, n4 - k4], [k9, n9 - k9]], alternative='less')[1]:.3g}")

**Result: not reproduced.** Flight day 4 ES maps (107, Rats 1 to 3) have a median of 1.214 bits/spike
against 1.441 for preflight (98); the one-sided Mann-Whitney p is 0.055, above the criterion's 0.05
(ratio 0.84, rank-biserial 0.129). The FD4 median is lower in each rat (ratios 0.65 to 0.88) but
significantly so only for Rat1 (p = 0.032), and the test passes with maps of peak rate 1 Hz or more
(p = 0.0095). FD4 is lower than FD9 for Rats 1 and 2 without significance (1.148 against 1.385, p =
0.21). The spike-based fraction of significant units is not lower on FD4 (21 of 71 against 16 of 53
on FD9, p = 0.55). The direction agrees with the claim; the criterion's test falls short. The claim's
"abnormal" may refer to something other than lower spatial information, which the description does
not say.

## fig3-stability-across-sessions

> Place fields are stable across sessions. (Figure 3 title; which sessions the figure compares is
> not known)

Criterion: Rat3 FD4, its two ES epochs. Same tetrode and cluster id in both, at least 50 spikes at
a known position in each. Smoothed 32 x 32 rate maps; median same-unit correlation against 1000
random re-pairings. Reproduced when above the 95th percentile of the re-paired medians and above
0.5. The cluster id is assumed to name the same neuron in both epochs (see the data check).

In [ ]:
rng_c4 = np.random.default_rng(72910)
s3 = by_label["Rat3 FD4"]
es_epochs = s3["epochs"][s3["epochs"].type == "ES"].sort_values("start")
lookup3 = nl.PositionLookup(s3)
maps3, occs = {}, []
for k, (a, b) in enumerate(zip(es_epochs.start, es_epochs.stop)):
    occ, scores = nl.epoch_occupancy(s3, a, b, "ES")
    occs.append(nl.merge_bins(nl.clean_occupancy(occ)))
    print(f"ES epoch {k + 1}: {a:.0f}-{b:.0f} s, occupancy map match {scores}")
    for _, u in s3["units"].iterrows():
        sp = u.spikes[(u.spikes >= a) & (u.spikes <= b)]
        found, r, c = lookup3(sp)
        if found.sum() >= nl.MIN_SPIKES:
            maps3[(u.tetrode, u.cluster, k)] = nl.smoothed_rate_map(r, c, occ)[0]
both = sorted({(t, c) for t, c, k in maps3 if (t, c, 0) in maps3 and (t, c, 1) in maps3})
only = sorted({(t, c) for t, c, k in maps3} - set(both))
print(f"Units with >= {nl.MIN_SPIKES} spikes in both epochs: {len(both)}; in one only: {len(only)}; units in file: {len(s3['units'])}")
shared = (occs[0] >= nl.MIN_OCC) & (occs[1] >= nl.MIN_OCC)
print("Bins with occupancy >= 5 in both epochs:", int(shared.sum()))

def corr(m1, m2):
    ok = shared & np.isfinite(m1) & np.isfinite(m2)
    return np.corrcoef(m1[ok], m2[ok])[0, 1]

A = [maps3[(t, c, 0)] for t, c in both]
B = [maps3[(t, c, 1)] for t, c in both]
C = np.array([[corr(a, b) for b in B] for a in A])
same = np.diag(C)
for (t, c), r in zip(both, same):
    print(f"  {t} cluster {c}: r = {r:.3f}")
null = np.array([np.median(C[np.arange(len(both)), rng_c4.permutation(len(both))]) for _ in range(1000)])
print(f"\nMedian same-unit r = {np.median(same):.3f} (range {same.min():.3f} to {same.max():.3f}); "
      f"re-paired medians: mean {null.mean():.3f}, 95th percentile {np.percentile(null, 95):.3f}; "
      f"p = {(np.sum(null >= np.median(same)) + 1) / 1001:.4f}")
off = C[~np.eye(len(both), dtype=bool)]
print(f"Different-unit pairs: median r {np.median(off):.3f}, range {off.min():.3f} to {off.max():.3f}")
print("Same-unit r above 0.5:", int((same > 0.5).sum()), "of", len(same), "; above 0.3:", int((same > 0.3).sum()))

print("\nVariants of the smoothing and the spike threshold (same re-pairing test, 1000 re-pairings each):")
rows = []
for sigma in [0.5, 1.0, 2.0]:
    for min_spikes in [50, 100]:
        mm = {}
        for k, (a, b) in enumerate(zip(es_epochs.start, es_epochs.stop)):
            occ, _ = nl.epoch_occupancy(s3, a, b, "ES")
            for _, u in s3["units"].iterrows():
                sp = u.spikes[(u.spikes >= a) & (u.spikes <= b)]
                found, r, c = lookup3(sp)
                if found.sum() >= min_spikes:
                    mm[(u.tetrode, u.cluster, k)] = nl.smoothed_rate_map(r, c, occ, sigma)[0]
        bb = sorted({(t, c) for t, c, k in mm if (t, c, 0) in mm and (t, c, 1) in mm})
        CC = np.array([[corr(mm[(t1, c1, 0)], mm[(t2, c2, 1)]) for t2, c2 in bb] for t1, c1 in bb])
        nn = np.array([np.median(CC[np.arange(len(bb)), rng_c4.permutation(len(bb))]) for _ in range(1000)])
        rows.append({"sigma_bins": sigma, "min_spikes": min_spikes, "units": len(bb), "median_r": round(float(np.median(np.diag(CC))), 3),
                     "null_95th": round(float(np.percentile(nn, 95)), 3), "p": round((np.sum(nn >= np.median(np.diag(CC))) + 1) / 1001, 4)})
print(pd.DataFrame(rows).to_string(index=False))

In [ ]:
# Per-session results, and the variant on the dandiset's own rate maps (ES1CEL~N against ES2CEL~N).
per_epoch = tested[(tested.rat == "Rat3") & (tested.day == "FD4") & (tested.type == "ES")]
print(per_epoch.groupby("epoch").agg(tested=("sig", "size"), significant=("sig", "sum"), median_info=("info", "median")).round(3).to_string())

mp = s3["maps"]
rows = []
for tt in sorted(mp.tetrode.unique()):
    for i in mp.index[(mp.tetrode == tt) & mp.source_file.str.startswith("ES1CEL")]:
        j = mp.index[(mp.tetrode == tt) & (mp.source_file == mp.loc[i, "source_file"].replace("ES1", "ES2"))]
        if len(j):
            r1, r2 = s3["rate_maps"][i], s3["rate_maps"][j[0]]
            ok = (r1 >= 0) & (r2 >= 0) & (s3["occupancy_maps"][i] > 0) & (s3["occupancy_maps"][j[0]] > 0)
            ok[nl.SENTINEL_BIN] = False
            rows.append({"tetrode": tt, "map": mp.loc[i, "source_file"], "r": np.corrcoef(r1[ok], r2[ok])[0, 1]})
rv = pd.DataFrame(rows)
print("\nDandiset rate maps, ES1CEL~N against ES2CEL~N on the same tetrode (the ~N match is an assumption):")
print(rv.round(3).to_string(index=False))
if len(rv):
    print(f"median r {rv.r.median():.3f} over {rv.r.notna().sum()} pairs with a value, of {len(rv)}")

**Result: reproduced.** In Rat3's two flight day 4 ES epochs, the 15 units with 50 or more spikes in
each have a median rate-map correlation of 0.538 (range -0.108 to 0.849), above the 95th percentile of
re-paired medians (0.091, p = 0.001) and above 0.5; 8 of 15 exceed 0.5. With other smoothing the
median is 0.312 to 0.704, always above its re-paired 95th percentile (p = 0.001), but at 0.5 bins of
smoothing it is below 0.5, which would make the claim partially reproduced. The dandiset's own maps
paired by their `~N` names give a median of 0.157, as those names do not identify cells. In the
second epoch fewer units are significant (2 of 19 against 9 of 16). The result assumes that a cluster
id names the same neuron in both epochs; which sessions Figure 3 compares is not known.

## theta-normal and ripples-normal

> Hippocampal EEG showed normal theta rhythm during locomotion and normal sharp waves and ripples
> during quiet rest in flight. (Dandiset description)

Blocked. No file holds EEG or any continuous voltage (see the listing in the data check). The NASA
record of the study (OSD-968) lists spike files, rate-map files and NWB copies, and no EEG.

## Summary

| Claim | Outcome |
|---|---|
| `flight-spatial-code` | Partially reproduced: Rat1 FD9 0.52 of units significant, Rat2 FD9 0.107 |
| `three-orthogonal-surfaces` | Blocked: metadata insufficient (no mapping of pixels to the track's surfaces) |
| `si-comparable-to-preflight` | Reproduced: FD9 1.385 against preflight 1.448 bits/spike, p = 0.45 |
| `fd4-abnormal-selectivity` | Not reproduced: FD4 1.214 against preflight 1.441 bits/spike, one-sided p = 0.055 |
| `fig3-stability-across-sessions` | Reproduced: median r 0.538 between Rat3's two FD4 ES epochs (null 95th percentile 0.091) |
| `theta-normal` | Blocked: data missing (no EEG) |
| `ripples-normal` | Blocked: data missing (no EEG) |

A failure to reproduce here may be a failure of this attempt, not of the paper: see the
reasoning and the differences from the paper's method.

How the analysis differs from the paper's. The paper's full text was not available, so its
Methods, tests and thresholds are unknown; every method here is a common choice fixed before the
analysis. Three claims come from the dandiset's description of the paper's findings, and one from a
figure title alone. Spatial information is computed from the spikes for the spike-based claims, with
the occupancy maps of the original analysis, as the dandiset holds position only at spike times;
the comparisons with preflight use the original rate maps, as the preflight files have no position.
Spatial information of the rate maps is not corrected for the number of spikes. The stability claim
assumes that a cluster id names the same neuron in two epochs.

What was found about the dandiset.
- The Rat1 and Rat2 files of each flight day hold the same position trace (99% of common samples
  within 2 pixels). Rat1's units are spatially selective with it; Rat2's are not, in any window, and
  in Rat2's FD4 ES epoch the trace is consistent with Rat1's position. Whose position the trace is
  in Rat2's task epochs is not stated and could not be established.
- Position is given only at spike times, so occupancy cannot be computed from the files' position.
- The rate-map `cell_number` comes from Windows 8.3 short file names (`ESCELL~1`, `ES2BC0~1`) and is
  not a cell number; it equals the cluster id of the best-matching unit in 84 of 211 unit-epochs, so
  the rate maps cannot be linked to units as the table's description says.
- Rate maps use -999 for unvisited bins, which the description does not state; the tracker's lost
  position (0, 255) is counted in the original occupancy (up to 19% in flight). Eight rate maps are
  identical copies of another map in the same file, some under a different tetrode.
- Units merge the per-epoch spike files with the same tetrode and cluster id; whether these are the
  same neuron is not stated. The unit table lacks the `session_type` and `source_file` columns the
  conversion's README lists.
- Spike times are given to 10 ms in five files; 2 to 15% of within-unit intervals are zero.
- There is no EEG, although the dandiset's description reports theta and ripples.